## drink AppleCider 



<u>what do you need?</u>

- preprocessed data from the [Bright Transient Survey](https://sites.astro.caltech.edu/ztf/bts/). the steps/formating are outlined in `001-data-processing.ipynb`. from that notebook specifically you'll need:
    - data_train_BTS.csv (`config[df_path]`) <small>(csv included in repo) </small>
    - preprocessed data (`config[preprocessed_path]`) for `data_train`, `data_test` <small>(<i>NOT</i> included in repo) </small>
    
- from DataGenerator:
    - train_files.pkl (`config[train_files_path]`), val_files.pkl (`config[val_files_path]`) created in `001-data-processing.ipynb`. <small>(included in repo) </small> alternatively, select `config[generate_train_val_files]: True`, will generate new train, val files their respective paths.
    - scaler_BTS.pkl (`config[scaler_path]`) created in `003-AppleCider-metadata.ipynb`.
    

In [ ]:
import os
import sys
sys.path.insert(0, 'AppleCider')

import numpy as np
import pandas as pd

from datetime import datetime
import wandb
from tqdm.auto import tqdm
import pickle

import AppleCider.core.drink as drink

In [ ]:
CLASSES = ['SN Ia', 'SN II', 'SN IIP', 'Cataclysmic', 'AGN', 'SN IIn', 'SN Ic', 'SN Ib', 'SN IIb', 'Tidal Disruption Event']

config = {
        'project': 'AppleCider',
        'mode': 'all',    # 'photo' 'spectra' 'meta' 'image' 'all'
        'config_from': None,
        'random_seed': 42,  # 42, 66, 0, 12, 123
        'use_wandb': True,
        'save_weights': False,
        'weights_path': f'/AppleCider/weights/{datetime.now().strftime("%Y-%m-%d-%H-%M")}',
        'use_pretrain': None,
        'freeze': False,

        ## Data General
        'preprocessed_path':'../data_train_BTS',
        'df_path': '../csv-pkl/data_train_BTS.csv',
        ## train, val files 
        'generate_train_val_files': False,
        'class_weights': False,
        'train_files_path': '../csv-pkl/train_files_BTS.pkl',
        'val_files_path': '../csv-pkl/val_files_BTS.pkl',
        'class_weights_path': '../csv-pkl/class_weights_BTS.pkl',
    
        ## Classes / tpes
        'step': 'type',
        'classes': CLASSES,
        'group_labels': False,
        'max_samples': 800,
        'num_classes': len(CLASSES),

        ## Photometry Model
        'seq_len': 230,
        'p_enc_in': 4,
        'p_d_model': 24,
        'p_dropout': 0.2,
        'p_factor': 1,
        'p_output_attention': False,
        'p_n_heads': 12,
        'p_d_ff': 512,
        'p_activation': 'gelu',
        'p_e_layers': 8,

        ## Spectra Model
        's_dropout': 0.2, 
        's_conv_channels': [1, 64, 64, 32, 32],
        's_kernel_size': 3,
        's_mp_kernel_size': 4,

        ## Metadata Model
        'm_hidden_dim': 512,
        'm_dropout': 0.2,
        'meta_cols': range(10),
        'scaler_path': '../csv-pkl/scaler.pkl',

        ## Image Model
        'input_channels': 3,
        'conv1_channels': 64,
        'conv2_channels': 16,
        'conv_kernel': 3,
        'conv_dropout1': 0.45,
        'conv_dropout2': 0.65,

        ## MultiModal Model
        'hidden_dim': 512,
        'fusion': 'avg',  # 'avg', 'concat'

        ## Training
        'batch_size': 512,
        'lr': 0.001,
        'beta1': 0.9,
        'beta2': 0.999,
        'weight_decay': 0.01,
        'epochs':25,
        'early_stopping_patience': 10,
        'scheduler': 'ReduceLROnPlateau',  # 'ExponentialLR', 'ReduceLROnPlateau'
        'gamma': 0.9,  # for ExponentialLR scheduler
        'factor': 0.3,  # for ReduceLROnPlateau scheduler
        'patience': 3,  # for ReduceLROnPlateau scheduler
        'warmup': False,
        'warmup_epochs': 100,
        'clip_grad': False,
        'clip_value': 5
    }


In [ ]:
wandb.init(
    project=config['project'],
    config=config)

In [ ]:
drink.run(config)